# Coral Reef AI — YOLO11n Detection Baseline

**Model:** YOLO11n  
**Epochs:** 10  
**Task:** Single-class object detection (`target_coral`)

Standalone Google Colab notebook. It mounts Drive, rebuilds the exact same Train/Validation/Test split from persistent `Detection_v1`, converts COCO → YOLO format, trains YOLO11n for 10 epochs, and evaluates Validation only. The internal Test split is intentionally left untouched.

In [ ]:
# CELL 1 — Install Ultralytics
!pip -q install -U ultralytics

In [ ]:
# CELL 2 — Mount Drive + environment
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch, ultralytics, platform

print('='*100)
print('ENVIRONMENT')
print('='*100)
print('Python      :', platform.python_version())
print('PyTorch     :', torch.__version__)
print('Ultralytics :', ultralytics.__version__)
print('CUDA        :', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory/(1024**3)
print('GPU         :', GPU_NAME)
print(f'VRAM        : {GPU_VRAM_GB:.2f} GB')

In [ ]:
# CELL 3 — Paths + source dataset check
from pathlib import Path
import json, os, shutil
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('/content/drive/MyDrive/Coral_Reef_AI_Project')
SOURCE_ROOT = PROJECT_ROOT/'01_Data'/'Final'/'Detection_v1'
SOURCE_IMAGES = SOURCE_ROOT/'images'
SOURCE_ANN = SOURCE_ROOT/'annotations'
YOLO_DATASET_ROOT = Path('/content/Coral_YOLO11_Detection_v1')
DATA_YAML = YOLO_DATASET_ROOT/'data.yaml'
EXPERIMENT_PROJECT = PROJECT_ROOT/'06_Experiments'/'YOLO11n_10Epochs'
RUN_NAME = 'Detection'
METADATA_DIR = SOURCE_ROOT/'metadata'
METADATA_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_MANIFEST_PATH = METADATA_DIR/'yolo11_adapter_manifest.csv'

required = {
    'train': (SOURCE_IMAGES/'train', SOURCE_ANN/'instances_train.json', 452, 1540),
    'validation': (SOURCE_IMAGES/'validation', SOURCE_ANN/'instances_validation.json', 100, 354),
    'test': (SOURCE_IMAGES/'test', SOURCE_ANN/'instances_test.json', 94, 304),
}
for split,(img_dir,ann_path,ei,eb) in required.items():
    if not img_dir.exists(): raise FileNotFoundError(img_dir)
    if not ann_path.exists(): raise FileNotFoundError(ann_path)
    with open(ann_path,'r',encoding='utf-8') as f: coco=json.load(f)
    ni,nb=len(coco['images']),len(coco['annotations'])
    cats=[c['name'] for c in coco['categories']]
    print(f'{split:<12} images={ni:<4} boxes={nb:<5} classes={cats}')
    assert ni==ei and nb==eb and cats==['target_coral']
print('STATUS: SOURCE_DETECTION_V1_OK')

In [ ]:
# CELL 4 — COCO → YOLO adapter for this fresh runtime
if YOLO_DATASET_ROOT.exists():
    shutil.rmtree(YOLO_DATASET_ROOT)
for sub in ['images/train','images/val','images/test','labels/train','labels/val','labels/test']:
    (YOLO_DATASET_ROOT/sub).mkdir(parents=True, exist_ok=True)

split_map = {
 'train': {'target':'train','img_dir':SOURCE_IMAGES/'train','ann':SOURCE_ANN/'instances_train.json','ei':452,'eb':1540},
 'validation': {'target':'val','img_dir':SOURCE_IMAGES/'validation','ann':SOURCE_ANN/'instances_validation.json','ei':100,'eb':354},
 'test': {'target':'test','img_dir':SOURCE_IMAGES/'test','ann':SOURCE_ANN/'instances_test.json','ei':94,'eb':304},
}
records=[]; total_boxes=0
for source_split,cfg in split_map.items():
    target=cfg['target']
    out_img=YOLO_DATASET_ROOT/'images'/target
    out_lbl=YOLO_DATASET_ROOT/'labels'/target
    with open(cfg['ann'],'r',encoding='utf-8') as f: coco=json.load(f)
    assert [c['name'] for c in coco['categories']]==['target_coral']
    imgs={int(x['id']):x for x in coco['images']}
    byimg={k:[] for k in imgs}
    for a in coco['annotations']: byimg[int(a['image_id'])].append(a)
    missing=[]; invalid=[]; split_boxes=0
    for image_id,img in imgs.items():
        fn=img['file_name']; W=float(img['width']); H=float(img['height'])
        src=cfg['img_dir']/fn; dst=out_img/fn; lp=out_lbl/f'{Path(fn).stem}.txt'
        if not src.exists(): missing.append(str(src)); continue
        try: os.symlink(src,dst); method='SYMLINK'
        except Exception: shutil.copy2(src,dst); method='COPY_FALLBACK'
        lines=[]
        for a in byimg[image_id]:
            assert int(a['category_id'])==1
            x,y,w,h=map(float,a['bbox'])
            ok=w>0 and h>0 and x>=0 and y>=0 and x+w<=W+1e-6 and y+h<=H+1e-6
            if not ok: invalid.append((fn,a['bbox'])); continue
            xc,yc,wn,hn=(x+w/2)/W,(y+h/2)/H,w/W,h/H
            vals=[xc,yc,wn,hn]
            if not (all(np.isfinite(v) for v in vals) and 0<=xc<=1 and 0<=yc<=1 and 0<wn<=1 and 0<hn<=1):
                invalid.append((fn,vals)); continue
            lines.append(f'0 {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}')
        lp.write_text('
'.join(lines)+('
' if lines else ''),encoding='utf-8')
        split_boxes += len(lines)
        records.append({'source_split':source_split,'yolo_split':target,'filename':fn,'source_image':str(src),'local_image':str(dst),'label_file':str(lp),'box_count':len(lines),'link_method':method})
    print(f'{source_split:<12} -> {target:<5} images={len(imgs):<4} boxes={split_boxes:<5} missing={len(missing)} invalid={len(invalid)}')
    assert len(imgs)==cfg['ei'] and split_boxes==cfg['eb'] and not missing and not invalid
    total_boxes += split_boxes
assert total_boxes==2198
pd.DataFrame(records).to_csv(ADAPTER_MANIFEST_PATH,index=False)

yaml_text = '
'.join([
    f'path: {YOLO_DATASET_ROOT}',
    'train: images/train',
    'val: images/val',
    'test: images/test',
    '',
    'names:',
    '  0: target_coral',
    ''
])
DATA_YAML.write_text(yaml_text,encoding='utf-8')
print('data.yaml:',DATA_YAML)
print('STATUS: YOLO11_DATASET_ADAPTER_READY')

In [ ]:
# CELL 5 — Strict Ultralytics dataset validation
from ultralytics.data.utils import check_det_dataset
checked=check_det_dataset(str(DATA_YAML))
assert checked['nc']==1 and checked['names'][0]=='target_coral'
expected={'train':(452,1540),'val':(100,354),'test':(94,304)}
for split,(ei,eb) in expected.items():
    images=sorted((YOLO_DATASET_ROOT/'images'/split).glob('*.jpg'))
    labels=sorted((YOLO_DATASET_ROOT/'labels'/split).glob('*.txt'))
    boxes=0
    for p in labels:
        lines=[x.strip() for x in p.read_text(encoding='utf-8').splitlines() if x.strip()]
        for line in lines:
            parts=line.split(); assert len(parts)==5 and int(parts[0])==0
            assert all(0.0<=float(x)<=1.0 for x in parts[1:])
        boxes += len(lines)
    print(f'{split:<6} images={len(images):<4} labels={len(labels):<4} boxes={boxes:<5}')
    assert len(images)==ei and len(labels)==ei and boxes==eb
print('STATUS: YOLO11_DATASET_VALIDATED')

In [ ]:
# CELL 6 — Configure and load YOLO11n
# 10-epoch run: close_mosaic=2 so Mosaic is available for epochs 1-8 and off for the final 2.
from ultralytics import YOLO
import json

EPOCHS=10; IMGSZ=640; BATCH=16; SEED=42
EXPERIMENT_PROJECT.mkdir(parents=True,exist_ok=True)
config={
 'experiment':'YOLO11n_10Epochs','task':'single_class_object_detection','model':'yolo11n.pt','class':'target_coral',
 'epochs':EPOCHS,'imgsz':IMGSZ,'batch':BATCH,'seed':SEED,'deterministic':True,'pretrained':True,
 'optimizer':'auto','amp':True,'close_mosaic':2,'train_images':452,'validation_images':100,'test_images':94,
 'train_boxes':1540,'validation_boxes':354,'test_boxes':304,'test_used_during_training':False,
 'ultralytics_version':ultralytics.__version__,'torch_version':torch.__version__,'gpu':GPU_NAME,'gpu_vram_gb':round(GPU_VRAM_GB,2)
}
with open(EXPERIMENT_PROJECT/'experiment_config.json','w',encoding='utf-8') as f: json.dump(config,f,indent=2,ensure_ascii=False)
print(json.dumps(config,indent=2,ensure_ascii=False))
model=YOLO('yolo11n.pt')
model.info()

In [ ]:
# CELL 7 — Train YOLO11n for exactly 10 epochs
import time
start=time.time()
results=model.train(
    data=str(DATA_YAML), epochs=10, imgsz=640, batch=16, device=0,
    pretrained=True, optimizer='auto', seed=42, deterministic=True,
    amp=True, close_mosaic=2, workers=2, cache=False,
    save=True, save_period=2, plots=True, patience=100,
    project=str(EXPERIMENT_PROJECT), name=RUN_NAME, exist_ok=True, verbose=True
)
elapsed=(time.time()-start)/60
RUN_DIR=Path(results.save_dir)
BEST_PT=RUN_DIR/'weights'/'best.pt'; LAST_PT=RUN_DIR/'weights'/'last.pt'
print(f'Training time: {elapsed:.2f} min')
print('Run dir:',RUN_DIR)
print('best.pt:', 'FOUND' if BEST_PT.exists() else 'NOT FOUND')
print('last.pt:', 'FOUND' if LAST_PT.exists() else 'NOT FOUND')
if not BEST_PT.exists(): raise RuntimeError('best.pt not found')
print('STATUS: YOLO11N_10_EPOCH_TRAINING_COMPLETE')

In [ ]:
# CELL 8 — Evaluate best.pt on Validation only
from ultralytics import YOLO
import json
best_model=YOLO(str(BEST_PT))
metrics=best_model.val(
    data=str(DATA_YAML), split='val', imgsz=640, batch=16, device=0,
    plots=True, save_json=False, project=str(EXPERIMENT_PROJECT/'Evaluation'),
    name='Validation', exist_ok=True, verbose=True
)
summary={
 'model':'YOLO11n','checkpoint':str(BEST_PT),'split':'validation',
 'mAP50_95':float(metrics.box.map),'mAP50':float(metrics.box.map50),'mAP75':float(metrics.box.map75),
 'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),
 'speed_ms_per_image':{k:float(v) for k,v in metrics.speed.items()}
}
with open(EXPERIMENT_PROJECT/'validation_metrics.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2,ensure_ascii=False)
print(f"mAP@50:95 : {summary['mAP50_95']:.4f}")
print(f"mAP@50    : {summary['mAP50']:.4f}")
print(f"mAP@75    : {summary['mAP75']:.4f}")
print(f"Precision : {summary['precision']:.4f}")
print(f"Recall    : {summary['recall']:.4f}")
print('Speed ms/image:',summary['speed_ms_per_image'])
print('Best checkpoint:',BEST_PT)
print('TEST SET WAS NOT EVALUATED.')
print('STATUS: YOLO11N_VALIDATION_COMPLETE')

In [ ]:
# CELL 9 — Display saved plots
from IPython.display import display, Image as IPImage
for p in [RUN_DIR/'results.png',RUN_DIR/'confusion_matrix.png',RUN_DIR/'confusion_matrix_normalized.png',RUN_DIR/'BoxPR_curve.png',RUN_DIR/'BoxF1_curve.png']:
    if p.exists():
        print(p.name)
        display(IPImage(filename=str(p)))

## Expected final status

```text
STATUS: YOLO11N_10_EPOCH_TRAINING_COMPLETE
STATUS: YOLO11N_VALIDATION_COMPLETE
```

Keep the 94-image internal Test split untouched until final model evaluation.